In [1]:
import sys
from pathlib import Path
import torch

from tracers.project import setup_project
from tracers.datasets.factory import build_datasets, build_loader_for_model
from tracers.pipelines.trainer_factory import build_trainer

PROJECT_ROOT = Path.cwd().parent
setup = setup_project(PROJECT_ROOT)

MODEL_KEY = "tracer_global_np"

device = setup["device"]
experiment_config = setup["experiment_config"]
hyperparams = setup["hyperparams"]
segment_frames = setup["segment_frames"]
batch_size = setup["batch_size"]

In [2]:
audio_dir, beatmap_dir, train_dataset, val_dataset, test_dataset, train_ids, val_ids, test_ids = build_datasets(
    PROJECT_ROOT,
    segment_frames=segment_frames,
    seed=42,
)

train_loader = build_loader_for_model(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    model_key=MODEL_KEY,
)
val_loader = build_loader_for_model(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    model_key=MODEL_KEY,
)
test_loader = build_loader_for_model(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    model_key=MODEL_KEY,
)

In [3]:
trainer, model = build_trainer(
    model_key=MODEL_KEY,
    hyperparams=hyperparams,
    experiment_config=experiment_config,
    device=device,
    checkpoint_path=(
        str(PROJECT_ROOT / experiment_config.resume_from)
        if experiment_config.resume_from is not None
        else None
    ),
    train_loader=train_loader,
    valid_loader=val_loader,
)

# Load checkpoint if specified in YAML
if experiment_config.resume_from is not None:
    resume_from = PROJECT_ROOT / experiment_config.resume_from
    trainer.load_checkpoint(resume_from)
    for g in trainer.optimizer.param_groups:
        g["lr"] = hyperparams["learning_rate"]

    print("learning_rate set to ", hyperparams["learning_rate"])

Loaded c:\Users\joshua\vsjycode\csc173\transformers\TRACERS\notebooks\checkpoints\latest.pt (epoch 40, best val loss 0.1490)
Loaded c:\Users\joshua\vsjycode\csc173\transformers\TRACERS\notebooks\checkpoints\latest.pt (epoch 40, best val loss 0.1490)
learning_rate set to  0.0001


In [4]:
trainer.fit()

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\joshua\_netrc.
wandb: Currently logged in as: joshuardh (joshuardh-mindanao-state-university) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Validation: 100%|██████████| 151/151 [00:29<00:00,  5.06it/s]



Epoch 41/60

TRAIN
Loss   | Total: 0.1272 | NP: 0.0097 | NS: 0.0593
Onset  | P: 0.8271 | R: 0.8299 | F1: 0.8285
Action | P: 0.9048 | R: 0.8805 | F1: 0.8925
Active Exact Accuracy: 0.4053
GT Active Rate: 5.03% | Predicted Active Rate: 4.90%

VAL
Loss   | Total: 0.1501 | NP: 0.0125 | NS: 0.0623
Onset  | P: 0.7574 | R: 0.7406 | F1: 0.7489
Action | P: 0.8504 | R: 0.8606 | F1: 0.8555
Active Exact Accuracy: 0.3774
GT Active Rate: 4.37% | Predicted Active Rate: 4.42%


Epochs:   5%|▌         | 1/20 [04:37<1:27:57, 277.78s/it]

Saved checkpoint: checkpoints\latest.pt


Epochs:  10%|█         | 2/20 [09:09<1:22:15, 274.17s/it]

... (34 outputs left out)


Epoch 59/60

TRAIN
Loss   | Total: 0.1094 | NP: 0.0083 | NS: 0.0513
Onset  | P: 0.8450 | R: 0.8568 | F1: 0.8508
Action | P: 0.9155 | R: 0.9016 | F1: 0.9085
Active Exact Accuracy: 0.4383
GT Active Rate: 4.99% | Predicted Active Rate: 4.92%

VAL
Loss   | Total: 0.1601 | NP: 0.0140 | NS: 0.0620
Onset  | P: 0.7664 | R: 0.7281 | F1: 0.7468
Action | P: 0.8499 | R: 0.8622 | F1: 0.8560
Active Exact Accuracy: 0.3843
GT Active Rate: 4.37% | Predicted Active Rate: 4.43%
Saved checkpoint: checkpoints\latest.pt


Epochs: 100%|██████████| 20/20 [1:31:21<00:00, 274.05s/it]



Epoch 60/60

TRAIN
Loss   | Total: 0.1083 | NP: 0.0082 | NS: 0.0508
Onset  | P: 0.8496 | R: 0.8589 | F1: 0.8542
Action | P: 0.9176 | R: 0.9037 | F1: 0.9106
Active Exact Accuracy: 0.4408
GT Active Rate: 5.00% | Predicted Active Rate: 4.93%

VAL
Loss   | Total: 0.1626 | NP: 0.0143 | NS: 0.0622
Onset  | P: 0.7527 | R: 0.7324 | F1: 0.7424
Action | P: 0.8501 | R: 0.8617 | F1: 0.8558
Active Exact Accuracy: 0.3792
GT Active Rate: 4.37% | Predicted Active Rate: 4.43%
Saved checkpoint: checkpoints\latest.pt


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
time/epoch_seconds,█▂▂▃▄▄▄▃▃▄▃▃▁▇▅▇▆▇▆▇
time/train_seconds,█▂▃▃▄▄▄▂▃▄▃▃▁▇▆▇▇▇▇▇
time/val_seconds,█▁▂▄▄▄▄▃▃▄▄▄▁▇▄▄▅▅▃▅
train/action_f1,▁▁▂▂▃▃▃▃▄▅▅▅▅▆▆▆▇▇▇█
train/action_precision,▁▂▂▂▃▄▄▃▄▅▅▅▅▆▆▆▇▇▇█
train/action_recall,▁▁▂▂▃▃▃▃▄▅▅▅▅▆▆▆▇█▇█
train/active_exact_accuracy,▁▁▂▂▃▃▃▃▄▅▅▅▆▆▆▆▇█▇█
train/gt_active_rate,▇█▂▃▆▃▄▂▄▄▅▃▄▅▁▃▆▆▄▅
train/np_loss,██▇▆▆▅▅▆▅▄▄▄▄▃▃▂▂▁▁▁
+18,...
